In [1]:
import ROOT
from references.constants import *
from utils.functions import get_mc_files
from pathlib import Path

print(SELECTIONS)

Welcome to JupyROOT 6.30/02
['SL_3j_resolved', 'SL_4j_resolved', 'SL_res_3j_1b', 'SL_res_3j_2b', 'SL_res_4j_1b', 'SL_res_4j_2b', 'SL_resolved', 'SL_boosted', 'DL_boosted', 'SL_res_1b', 'SL_res_2b', 'DL_res_1b', 'DL_res_2b', 'baseSel', '_noSel', 'noSel', 'Total']


In [2]:
def find_matching_selection(hist_name, selections):
    """
    Find which selection name matches the beginning of the histogram name.
    Uses the sorted list (longest first) to ensure most specific matches.
    Only processes histograms that haven't been renamed yet (don't contain ___).
    """
    # Skip histograms that already have the new format
    if CHANNEL_DISCRIMINANT_DELIM in hist_name:
        return None
        
    for selection in selections:
        if hist_name.startswith(selection + "_"):
            return selection
    return None

def rename_histograms_in_file(file_path, dry_run=True, debug=False):
    """
    Rename histograms in a ROOT file from {selection}_{variable} to {selection}___{variable}
    
    Args:
        file_path: Path to the ROOT file
        dry_run: If True, only print what would be renamed without actually doing it
        debug: If True, show all histogram names and matching process
    """
    print(f"\nProcessing file: {file_path}")
    
    # Open file in UPDATE mode to modify it
    if not dry_run:
        root_file = ROOT.TFile.Open(str(file_path), "UPDATE")
    else:
        root_file = ROOT.TFile.Open(str(file_path), "READ")
    
    if not root_file or root_file.IsZombie():
        print(f"ERROR: Could not open {file_path}")
        return False
    
    # Get list of all keys (histogram names)
    keys = root_file.GetListOfKeys()
    histograms_to_rename = []
    all_hist_names = []
    skipped_already_renamed = []
    skipped_no_match = []
    
    # First pass: identify histograms that need renaming
    for key in keys:
        hist_name = key.GetName()
        hist_class = key.GetClassName()
        
        # Only process histograms (TH1, TH2, etc.)
        if not hist_class.startswith("TH"):
            continue
            
        all_hist_names.append(hist_name)
        
        # Check if already renamed
        if CHANNEL_DISCRIMINANT_DELIM in hist_name:
            skipped_already_renamed.append(hist_name)
            continue
            
        # Find matching selection
        matching_selection = find_matching_selection(hist_name, SELECTIONS)
        
        if matching_selection:
            # Extract variable part (everything after selection_)
            variable_part = hist_name[len(matching_selection) + 1:]
            new_name = f"{matching_selection}{CHANNEL_DISCRIMINANT_DELIM}{variable_part}"
            
            if new_name != hist_name:  # Only if name actually changes
                histograms_to_rename.append((hist_name, new_name))
        else:
            skipped_no_match.append(hist_name)
    
    if debug:
        print(f"\nDEBUG INFO:")
        print(f"Total histograms found: {len(all_hist_names)}")
        print(f"Already renamed (contain '{CHANNEL_DISCRIMINANT_DELIM}'): {len(skipped_already_renamed)}")
        print(f"No selection match: {len(skipped_no_match)}")
        print(f"To be renamed: {len(histograms_to_rename)}")
        
        if skipped_no_match:
            print(f"\nHistograms with no selection match (first 10):")
            for name in skipped_no_match[:10]:
                print(f"  {name}")
        
        # Show SL_res_2b specifically
        sl_res_2b_hists = [name for name in all_hist_names if 'SL_res_2b' in name]
        if sl_res_2b_hists:
            print(f"\nFound {len(sl_res_2b_hists)} histograms containing 'SL_res_2b':")
            for name in sl_res_2b_hists[:5]:  # Show first 5
                print(f"  {name}")
    
    print(f"Found {len(histograms_to_rename)} histograms to rename:")
    
    # Show what will be renamed
    for old_name, new_name in histograms_to_rename:
        print(f"  {old_name} -> {new_name}")
    
    if dry_run:
        print("DRY RUN: No changes made to file")
        root_file.Close()
        return True
    
    # Second pass: actually rename the histograms
    renamed_count = 0
    for old_name, new_name in histograms_to_rename:
        hist = root_file.Get(old_name)
        if hist:
            # Set new name
            hist.SetName(new_name)
            hist.SetTitle(new_name)  # Also update title to match name
            
            # Write with new name
            hist.Write(new_name, ROOT.TObject.kOverwrite)
            
            # Delete old histogram
            root_file.Delete(f"{old_name};*")  # ;* deletes all cycles
            
            renamed_count += 1
        else:
            print(f"WARNING: Could not retrieve histogram {old_name}")
    
    # Save changes
    root_file.Write()
    root_file.Close()
    
    print(f"Successfully renamed {renamed_count} histograms in {file_path}")
    return True

def process_root_files(file_paths, dry_run=True, debug=False):
    """
    Process multiple ROOT files
    
    Args:
        file_paths: List of paths to ROOT files or glob patterns
        dry_run: If True, only show what would be renamed
        debug: If True, show detailed debugging information
    """
    processed_files = 0
    failed_files = 0
    
    for file_pattern in file_paths:
        # Handle glob patterns
        if isinstance(file_pattern, str):
            if '*' in file_pattern or '?' in file_pattern:
                files = list(Path('.').glob(file_pattern))
            else:
                files = [Path(file_pattern)]
        else:
            files = [Path(file_pattern)]
        
        for file_path in files:
            if file_path.suffix.lower() == '.root':
                if rename_histograms_in_file(file_path, dry_run, debug):
                    processed_files += 1
                else:
                    failed_files += 1
            else:
                print(f"Skipping non-ROOT file: {file_path}")
    
    print(f"\nSummary:")
    print(f"  Files processed: {processed_files}")
    print(f"  Files failed: {failed_files}")

In [3]:
workdir = Path('/eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_New/JetTop_even')
resultsdir = workdir / 'results'
root_files = get_mc_files(resultsdir)

# First do a dry run to see what would be changed
print("=== DRY RUN ===")
process_root_files(root_files, dry_run=True)

# Ask for confirmation
response = input("\nDo you want to proceed with renaming? (y/N): ")
if response.lower() in ['y', 'yes']:
    print("\n=== ACTUAL RENAMING ===")
    process_root_files(root_files, dry_run=False)
else:
    print("Operation cancelled.")

=== DRY RUN ===

Processing file: /eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_New/JetTop_even/results/ggHH_kl_1_kt_1_bbww_dl_2022.root
Found 58 histograms to rename:
  SL_res_1b_bjet0_pt -> SL_res_1b___bjet0_pt
  SL_res_1b_all_sT -> SL_res_1b___all_sT
  SL_res_1b_all_mInv -> SL_res_1b___all_mInv
  SL_res_1b_all_mT -> SL_res_1b___all_mT
  SL_res_1b_all_jets_HT -> SL_res_1b___all_jets_HT
  SL_res_1b_all_pt -> SL_res_1b___all_pt
  SL_res_1b_lep0_pt -> SL_res_1b___lep0_pt
  SL_res_1b_lep0_eta -> SL_res_1b___lep0_eta
  SL_res_1b_lep0_phi -> SL_res_1b___lep0_phi
  SL_res_1b_lep0_iso -> SL_res_1b___lep0_iso
  SL_res_1b_ak4_jet0_pt -> SL_res_1b___ak4_jet0_pt
  SL_res_1b_ak4_jet0_eta -> SL_res_1b___ak4_jet0_eta
  SL_res_1b_ak4_jet0_phi -> SL_res_1b___ak4_jet0_phi
  SL_res_1b_ak4_jet0_bscore -> SL_res_1b___ak4_jet0_bscore
  SL_res_1b_ak4_jet1_pt -> SL_res_1b___ak4_jet1_pt
  SL_res_1b_ak4_jet1_eta -> SL_res_1b___ak4_jet1_eta
  SL_res_1b_ak4_jet1_phi -> SL_res_1b___ak4_jet1_phi
  SL_res_1b_ak4_je

Error in <TNetXNGFile::Close>: [ERROR] Socket timeout
